In [1]:
# imporr the libraries
import numpy as np
import pandas as pd
import difflib # for comparing sequences
from sklearn.feature_extraction.text import TfidfVectorizer # for converting text to matrix of TF-IDF features
from sklearn.metrics.pairwise import cosine_similarity # to compute cosine similarity between samples in two matrices

In [2]:
# read data
song_data = pd.read_csv("./spotify_millsongdata.csv")
song_data.head()

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \nAnd..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \nTouch me gentl..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \nWhy I had t...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [3]:
song_data.shape

(57650, 4)

In [4]:
# get the length of text feature
song_data["lyrics_word_count"] = (
    song_data["text"]
    .fillna("")
    .str.split()
    .str.len()
)

avg_words = song_data["lyrics_word_count"].mean()
max_words = song_data["lyrics_word_count"].max()
median_words = song_data["lyrics_word_count"].median()

print(avg_words, median_words, max_words)



219.48626192541198 196.0 827


In [5]:
song_data.isnull().sum() # not missing values

artist               0
song                 0
link                 0
text                 0
lyrics_word_count    0
dtype: int64

Their appears to be no missing values in the dataset

In [6]:
song_data.columns

Index(['artist', 'song', 'link', 'text', 'lyrics_word_count'], dtype='object')

All the features are object data types

In [7]:
song_data.dtypes

artist               object
song                 object
link                 object
text                 object
lyrics_word_count     int64
dtype: object

In [8]:
song_data.describe(include="object") # summary statistics for the categorical features

,artist,song,link,text
count,57650,57650,57650,57650
unique,643,44824,57650,57494
top,Donna Summer,Have Yourself A Merry Little Christmas,/z/zwan/heartsong_20148991.html,I just came back from a lovely trip along the ...
freq,191,35,1,6


The Artist Donna Summer is the artist with most songs in the dataset with frequency of 191

In [9]:
# Selecting features for the recommendation engine
features = ['artist', 'song', 'text'] # this are the 3 features of interest



In [ ]:
# data Cleaning
# handing missing values


In [10]:
# Creating a new combined features column
combined_features = song_data[features].apply(lambda row: ' '.join(row), axis=1)

In [11]:
# vectorizing the combined features using bounded TFIDF -
# this is important has the dataset is large over 57k.
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=25_000,   # reduced due to moderate text length
    min_df=3,
    max_df=0.85,
    ngram_range=(1, 2),
    sublinear_tf=True
)

feature_vectors = tfidf.fit_transform(combined_features)


Some settings tune for the tfidf are

* Vocabulary capped to prevent memory spikes

* Bigrams capture phrase semantics

* Sublinear TF stabilizes long lyrics

In [12]:
print(feature_vectors)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4014310 stored elements and shape (57650, 25000)>
  Coords	Values
  (0, 59)	0.1408246904807611
  (0, 10855)	0.20529926301532886
  (0, 7415)	0.15939548480182286
  (0, 13217)	0.09587552991638981
  (0, 6113)	0.10562919257711921
  (0, 24328)	0.11187021218440683
  (0, 14609)	0.10033421944867846
  (0, 19940)	0.10401969598901274
  (0, 23659)	0.04493737060221901
  (0, 19635)	0.11540971161990923
  (0, 18924)	0.1121669533165933
  (0, 13991)	0.10310232286962355
  (0, 6560)	0.13870154862308168
  (0, 10444)	0.08297601262489347
  (0, 14238)	0.1302059178568248
  (0, 6379)	0.08627061656321465
  (0, 6649)	0.13934503595478936
  (0, 1392)	0.11168599976234889
  (0, 1860)	0.12523166728878607
  (0, 11634)	0.1687882477258094
  (0, 23198)	0.06941685199167141
  (0, 16392)	0.10691627833145695
  (0, 9601)	0.11706347864022763
  (0, 8645)	0.0661541626250361
  (0, 12787)	0.038785015072743
  :	:
  (57649, 9059)	0.08864433523286394
  (57649, 18054)	0.08430

In [13]:
# getting the shape of the feature vectors
feature_vectors.shape

(57650, 25000)

In [14]:
# this is high dimension matrix - lets reduce the dimension using dimensionality reduction techniques 
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(
    n_components=200,
    random_state=42
)

feature_vectors_reduced = svd.fit_transform(feature_vectors)



The above compresses tens of thousands of sparse features into dense semantic vectors.

In [15]:
feature_vectors_reduced.shape

(57650, 200)

This reduces the feature vectors to a more manageable size while retaining most of the important information.

In [17]:
# fitting the similarity index using NearestNeighbors - using the metric cosine
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(
    n_neighbors=15,
    metric="cosine",
    algorithm="brute"
)

nn.fit(feature_vectors_reduced)


,n_neighbors,15
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [28]:
def recommend_from_song(song_title, k=10):
    # Find the song index
    matches = song_data[song_data["song"].str.lower() == song_title.lower()]
    
    if matches.empty:
        raise ValueError("Song not found in dataset")

    song_index = matches.index[0]

    # Find similar songs
    distances, indices = nn.kneighbors(
        feature_vectors_reduced[song_index].reshape(1, -1),
        n_neighbors=k + 1
    )

    # Exclude the input song itself
    rec_indices = indices[0][1:]

    return song_data.loc[rec_indices, ["song"]]


In [19]:
# 1 means 100% similar
# take the title of the movie from the user
list_of_all_song = song_data['song'].tolist()
list_of_all_song

["Ahe's My Kind Of Girl",
 'Andante, Andante',
 'As Good As New',
 'Bang',
 'Bang-A-Boomerang',
 'Burning My Bridges',
 'Cassandra',
 'Chiquitita',
 'Crazy World',
 'Crying Over You',
 'Dance',
 'Dancing Queen',
 'Disillusion',
 'Does Your Mother Know',
 'Dream World',
 'Dum Dum Diddle',
 'Eagle',
 'Every Good Man',
 'Fernando',
 'Fernando (In Spanish)',
 'Free As A Bumble Bee',
 'From A Twinkling Star To A Passing Angel',
 'Gimme Gimme Gimme',
 "Givin' A Little Bit More",
 'Gonna Sing You My Lovesong',
 'Hamlet III',
 'Happy Hawaii',
 'Happy New Year',
 'He Is Your Brother',
 'Head Over Heels',
 "Here We'll Stay",
 'Hey Hey Helen',
 'Hole In Your Soul',
 'Honey, Honey',
 'I Am Just A Girl',
 'I Am The City',
 'I Do, I Do, I Do, I Do, I Do',
 'I Have A Dream',
 'I Let The Music Speak',
 'I Saw It In The Mirror',
 'I Wonder (Departure)',
 'I Wonder (Departure) [Live]',
 "If It Wasn't For The Nights",
 "I'm A Marionette",
 "I've Been Waiting For You",
 'Juper Jrouper',
 'Just A Notion',


In [ ]:
# Recommends top 10 songs similar to "Siberia"
print(recommend_from_song("Siberia", k=10))


                      song
6872   I Can't Go For That
29710         Baby It's Me
17109           Then I Did
41436     Someone Like You
27711             Memories
37726     Time In A Bottle
35478         Did You Know
29951  Dreams Do Come True
45677     I'd Come For You
19922      It's My Delight


In [ ]:
# checking wether thier is duplicate song titles
# how will I know 
duplicate_songs = song_data[song_data.duplicated(subset=['song'], keep=False)]
duplicate_songs_sorted = duplicate_songs.sort_values(by='song')

In [21]:
duplicate_songs_sorted

,artist,song,link,text,lyrics_word_count
50299,Ray Charles,'Deed I Do,/r/ray+charles/deed+i+do_20618784.html,"\nDo I want you? \nOh my do I \nHoney, ind...",88
30995,Ella Fitzgerald,'Deed I Do,/e/ella+fitzgerald/deed+i+do_20813472.html,"Do I want you? \nOh my do I \nHoney, indeed ...",88
10249,Kelly Clarkson,(Love Will) Turn Back The Hands Of Time,/k/kelly+clarkson/love+will+turn+back+the+hand...,[Kelly] \nNo more midnight rides with you \n...,267
7197,Grease,(Love Will) Turn Back The Hands Of Time,/g/grease/love+will+turn+back+the+hands+of+tim...,[Stephanie] \nNo more midnight rides with you...,266
4722,Electric Light Orchestra,10538 Overture,/e/electric+light+orchestra/10538+overture_200...,"Did you see your friend, crying from his eyes ...",98
...,...,...,...,...,...
4829,Ellie Goulding,Your Song,/e/ellie+goulding/your+song_20895401.html,It's a little bit funny \nThis feeling inside...,207
5203,Engelbert Humperdinck,Yours,/e/engelbert+humperdinck/yours_20504122.html,Yours till the stars have no glory \nYours ti...,106
55005,Vera Lynn,Yours,/v/vera+lynn/yours_20354375.html,Yours 'til the stars lose their glory \nYours...,99
4762,Electric Light Orchestra,Zing Went The Strings Of My Heart,/e/electric+light+orchestra/zing+went+the+stri...,"Dear, when you smile at me \nI hear a melody ...",105
